# Composer Vector - Usando VAE para criar música no estilo de certos compositores

## 1. Imports e Configurações

In [ ]:
# Bibliotecas utilizadas

import sys
import json
import random
import re
from pathlib import Path
from contextlib import nullcontext
from collections import Counter
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, random_split
from miditok import REMI, TokenizerConfig, TokSequence
from mido import MidiFile


/home/al.thiago.oliveira/ml2/ml2_trabalhos_2026/env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Configurações de seed e otimização de algumas funções
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

output_dir = Path("trabalho_midi_outputs/ComposerVector")
output_dir.mkdir(parents=True, exist_ok=True)

## 2. Dataset

In [ ]:
## 2. DATASET
DATASET_DIR = Path("surname_checked_midis")

midi_paths = list(DATASET_DIR.glob("*.mid"))

contagem_compositores = {}
for path in midi_paths:
    sobrenome = path.name.split(",")[0].strip()
    contagem_compositores[sobrenome] = contagem_compositores.get(sobrenome, 0) + 1

# Mapeamento de alvos
compositores_disponiveis = sorted(list(contagem_compositores.keys()))

composer_to_id = {nome: idx for idx, nome in enumerate(compositores_disponiveis)}
id_to_composer = {v: k for k, v in composer_to_id.items()}
NUM_COMPOSERS = len(composer_to_id)

## 3. Tokenizador

In [ ]:
# Tokenizador otimizado para Piano Solo
config = TokenizerConfig(num_velocities=32, use_chords=True, use_programs=False)
tokenizer = REMI(config)

# Treinamento do Tokenizador BPE
tokenizer.train(vocab_size=3000, files_paths=midi_paths)
tokenizer.save(output_dir / "tokenizer_config.json")

X_tokens, y_labels = [], []
sequence_length = 1024
stride = 1024

print("Processamento dos arquivos MIDI...")
for path in midi_paths:
    sobrenome = path.name.split(",")[0].strip()
    composer_id = composer_to_id[sobrenome]
    try:
        tokens = tokenizer(path)[0].ids
        for i in range(0, len(tokens) - sequence_length, stride):
            X_tokens.append(tokens[i : i + sequence_length])
            y_labels.append(composer_id)
    except Exception:
        continue

X_tokens = np.array(X_tokens, dtype=np.int64)
y_labels = np.array(y_labels, dtype=np.int64)
print(f"Fragmentos musicais gerados: {X_tokens.shape[0]}")

class TransformerMusicDataset(Dataset):
    def __init__(self, x_data, y_data):
        self.X = torch.tensor(x_data, dtype=torch.long)
        self.y = torch.tensor(y_data, dtype=torch.long)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        sequence = self.X[idx]
        composer = self.y[idx]
        return sequence[:-1], sequence[1:], composer

dataset_completo = TransformerMusicDataset(X_tokens, y_labels)
val_size = int(len(dataset_completo) * 0.1)
train_size = len(dataset_completo) - val_size
train_ds, val_ds = random_split(dataset_completo, [train_size, val_size])

BATCH_SIZE = 32
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True, num_workers=4, pin_memory=True if device.type == "cuda" else False)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True if device.type == "cuda" else False)





Processamento dos arquivos MIDI...
Fragmentos musicais gerados: 50177


## 4. Arquitetura do Music Transformer

In [ ]:
VOCAB_SIZE = len(tokenizer)
MAX_SEQ_LEN = X_tokens.shape[1]
D_MODEL, N_HEADS, N_LAYERS, FF_DIM, DROPOUT = 512, 8, 6, 2048, 0.1

# Aplica o RoPE nos nossos tokens

def precompute_freqs_cis(dim: int, end: int, theta: float = 10000.0):
    freqs = 1.0 / (theta ** (torch.arange(0, dim, 2)[: (dim // 2)].float() / dim))
    t = torch.arange(end, device=freqs.device, dtype=torch.float32)
    freqs = torch.outer(t, freqs).float()
    return torch.polar(torch.ones_like(freqs), freqs)

def apply_rope(xq, xk, freqs_cis):
    xq_ = torch.view_as_complex(xq.float().reshape(*xq.shape[:-1], -1, 2))
    xk_ = torch.view_as_complex(xk.float().reshape(*xk.shape[:-1], -1, 2))
    freqs_cis = freqs_cis.view(1, 1, freqs_cis.shape[0], freqs_cis.shape[1])
    xq_out = torch.view_as_real(xq_ * freqs_cis).flatten(3)
    xk_out = torch.view_as_real(xk_ * freqs_cis).flatten(3)
    return xq_out.type_as(xq), xk_out.type_as(xk)

# Flash Attention
class CustomAttentionBlock(nn.Module):
    def __init__(self, d_model, n_heads, dropout):
        super().__init__()
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.qkv = nn.Linear(d_model, d_model * 3, bias=False)
        self.proj = nn.Linear(d_model, d_model, bias=False)
        self.dropout_p = dropout

    def forward(self, x, freqs_cis):
        B, T, C = x.shape
        qkv = self.qkv(x)
        qkv = qkv.reshape(B, T, 3, self.n_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]

        q, k = apply_rope(q, k, freqs_cis)

        dropout_rate = self.dropout_p if self.training else 0.0
        attn_out = F.scaled_dot_product_attention(q, k, v, is_causal=True, dropout_p=dropout_rate)

        attn_out = attn_out.transpose(1, 2).reshape(B, T, C)
        return self.proj(attn_out)

class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, ff_dim, dropout):
        super().__init__()
        self.attn = CustomAttentionBlock(d_model, n_heads, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(nn.Linear(d_model, ff_dim), nn.GELU(), nn.Dropout(dropout), nn.Linear(ff_dim, d_model))
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, freqs_cis):
        attn_out = self.attn(self.norm1(x), freqs_cis)
        x = x + self.dropout(attn_out)
        x = x + self.dropout(self.ff(self.norm2(x)))
        return x

class MusicTransformer(nn.Module):
    def __init__(self, vocab_size, d_model, max_seq_len, n_heads, n_layers, ff_dim, dropout):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, d_model)

        freqs_cis = precompute_freqs_cis(d_model // n_heads, max_seq_len * 2)
        self.register_buffer("freqs_cis", freqs_cis)

        self.layers = nn.ModuleList([TransformerBlock(d_model, n_heads, ff_dim, dropout) for _ in range(n_layers)])
        self.final_norm = nn.LayerNorm(d_model)
        self.output_head = nn.Linear(d_model, vocab_size, bias=False)

        self.token_embedding.weight = self.output_head.weight
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, steering_vector=None, steering_layer=-1, alpha=1.0):
        B, T = x.shape
        h = self.dropout(self.token_embedding(x))
        freqs_cis = self.freqs_cis[:T]

        target_layer_idx = len(self.layers) - 1 if steering_layer == -1 else steering_layer

        for idx, layer in enumerate(self.layers):
            h = layer(h, freqs_cis)

            # Verifica se chegamos na última camada para fazer steering
            if steering_vector is not None and idx == target_layer_idx:
                s_vector = steering_vector.unsqueeze(0).unsqueeze(1)
                h_norm_orig = torch.linalg.vector_norm(h, dim=-1, keepdim=True)
                h_steered = h + (alpha * s_vector)
                h_norm_steered = torch.linalg.vector_norm(h_steered, dim=-1, keepdim=True)
                h = h_steered * (h_norm_orig / (h_norm_steered + 1e-8))

        return self.output_head(self.final_norm(h))

    @torch.no_grad()
    def extract_last_hidden_state(self, x, target_layer=-1):
        B, T = x.shape
        h = self.dropout(self.token_embedding(x))
        freqs_cis = self.freqs_cis[:T]

        target_layer_idx = len(self.layers) - 1 if target_layer == -1 else target_layer

        for idx, layer in enumerate(self.layers):
            h = layer(h, freqs_cis)
            if idx == target_layer_idx:
                return h[:, -1, :]

model = MusicTransformer(VOCAB_SIZE, D_MODEL, MAX_SEQ_LEN, N_HEADS, N_LAYERS, FF_DIM, DROPOUT).to(device)



In [ ]:
# Configuração do Engine
EPOCHS = 100
PATIENCE = 5

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)
scaler = torch.amp.GradScaler(device.type, enabled=(device.type == "cuda"))

autocast_context = torch.autocast(device_type="cuda", dtype=torch.float16) if device.type == "cuda" else nullcontext()

def transformer_loss(logits, target):
    return F.cross_entropy(logits.reshape(-1, logits.size(-1)), target.reshape(-1))

start_epoch = 1
best_val_loss = float('inf')
history = {'train_loss': [], 'val_loss': [], 'val_perplexity': [], 'learning_rate': [], 'gpu_memory': []}

# Busca automática pelo último checkpoint
checkpoint_files = list(output_dir.glob("checkpoint_epoch_*.pt"))
if checkpoint_files:
    epochs_found = [int(re.findall(r"checkpoint_epoch_(\d+).pt", f.name)[0]) for f in checkpoint_files]
    latest_epoch = max(epochs_found)
    latest_checkpoint_path = output_dir / f"checkpoint_epoch_{latest_epoch}.pt"

    print(f"[CARREGAMENTO] O modelo mais recente (Época {latest_epoch}) foi encontrado.")
    print(f"Carregando pesos de: {latest_checkpoint_path.name} ...")

    checkpoint = torch.load(latest_checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    scheduler.load_state_dict(checkpoint['scheduler_state_dict'])

    if device.type == "cuda" and 'scaler_state_dict' in checkpoint:
        scaler.load_state_dict(checkpoint['scaler_state_dict'])

    start_epoch = checkpoint['epoch'] + 1
    best_val_loss = checkpoint.get('best_val_loss', float('inf'))

    if (output_dir / "training_history.json").exists():
        with open(output_dir / "training_history.json", "r") as f:
            history = json.load(f)

    print(f"✓ Modelo pronto! Se desejar treinar, a execução continuará da Época {start_epoch}.")
else:
    print("Nenhum checkpoint encontrado. O modelo está inicializado do zero.")

[CARREGAMENTO] O modelo mais recente (Época 93) foi encontrado.
Carregando pesos de: checkpoint_epoch_93.pt ...
✓ Modelo pronto! Se desejar treinar, a execução continuará da Época 94.


In [ ]:
# Definição do Loop de Treinamento
def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss, total_samples = 0.0, 0
    for step, (x, target, _) in enumerate(loader):
        x, target = x.to(device), target.to(device)
        if train:
            optimizer.zero_grad(set_to_none=True)
            with autocast_context:
                logits = model(x)
                loss = transformer_loss(logits, target)
            if not torch.isfinite(loss):
                continue
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.5)
            if torch.isfinite(grad_norm):
                scaler.step(optimizer)
                scaler.update()
            else:
                optimizer.zero_grad(set_to_none=True)
                scaler.update()
        else:
            with torch.no_grad():
                with autocast_context:
                    logits = model(x)
                    loss = transformer_loss(logits, target)
        total_loss += loss.item() * x.size(0)
        total_samples += x.size(0)
    return total_loss / total_samples if total_samples > 0 else None

In [ ]:

# Loop de Execução
# print("--- Iniciando Loop de Treinamento Global (V3 - Todos os Compositores) ---")
# patience_counter = 0

# for epoch in range(start_epoch, EPOCHS + 1):
#     tr_loss = run_epoch(train_loader, train=True)
#     val_loss = run_epoch(val_loader, train=False)
#     if tr_loss is None or val_loss is None:
#         sys.exit(1)

#     val_perplexity = np.exp(val_loss)
#     current_lr = optimizer.param_groups[0]['lr']
#     gpu_mem_used = torch.cuda.memory_allocated(device) / (1024**3) if device.type == "cuda" else 0.0

#     history['train_loss'].append(tr_loss)
#     history['val_loss'].append(val_loss)
#     history['val_perplexity'].append(val_perplexity)
#     history['learning_rate'].append(current_lr)
#     history['gpu_memory'].append(gpu_mem_used)

#     print(f"Epoch {epoch:03d}/{EPOCHS} | Train Loss: {tr_loss:.4f} | Val Loss: {val_loss:.4f} | Val PPL: {val_perplexity:.2f}")

#     scheduler.step(val_loss)

#     if val_loss < best_val_loss:
#         best_val_loss = val_loss
#         patience_counter = 0
#         checkpoint = {
#             'epoch': epoch,
#             'model_state_dict': model.state_dict(),
#             'optimizer_state_dict': optimizer.state_dict(),
#             'scheduler_state_dict': scheduler.state_dict(),
#             'scaler_state_dict': scaler.state_dict() if device.type == "cuda" else None,
#             'best_val_loss': best_val_loss
#         }
#         torch.save(checkpoint, output_dir / f"checkpoint_epoch_{epoch}.pt")
#         with open(output_dir / "training_history.json", "w") as f:
#             json.dump(history, f, indent=4)
#     else:
#         patience_counter += 1
#         if patience_counter >= PATIENCE:
#             print(f"\n[EARLY STOPPING] Treinamento interrompido por falta de melhoria na Época {epoch}.")
#             break



## 5. Extração do Composer Vector

In [ ]:
def build_composer_vectors(model, dataloader, target_layer=-1):
    model.eval()
    embeddings = {comp_id: [] for comp_id in range(NUM_COMPOSERS)}
    with torch.no_grad():
        for x, _, composer_labels in dataloader:
            x = x.to(device)
            h_T = model.extract_last_hidden_state(x, target_layer=target_layer)
            for emb, comp_id in zip(h_T.cpu().numpy(), composer_labels.numpy()):
                embeddings[comp_id].append(emb)
    composer_vectors = {}
    for comp_id, embs in embeddings.items():
        if len(embs) > 0:
            composer_vectors[comp_id] = np.mean(embs, axis=0)
    return composer_vectors, embeddings

print("\nGerando os Centróides de Estilo Globais (Todos os Compositores)...")
composer_vectors, embeddings = build_composer_vectors(model, train_loader)
torch.save(composer_vectors, output_dir / "composer_vectors.pt")
print("✓ Vetores de Estilo salvos com sucesso na nova pasta!")


Gerando os Centróides de Estilo Globais (Todos os Compositores)...
✓ Vetores de Estilo salvos com sucesso na nova pasta!


Chopin: 309

Beethoven: 112

Bach: 67

## 6. Geração do MIDI

In [ ]:
COMPOSER_ID = composer_to_id["Beethoven"]
ALPHA = 0.2              # Intensidade do steering
termo = "Abreu, Zequinha, Tico-Tico no fubá, KyEuFpJuwfc.mid"
SEED_INDEX = [(print(f"SEED_INDEX: {idx} -> {path.name}"), idx)[1] for idx, path in enumerate(midi_paths) if termo.lower() in path.name.lower()][0]

# Índice da música do dataset para iniciar
MAX_NEW_TOKENS = 1024     # Quantos tokens gerar
TEMPERATURE= 0.95
OUTPUT_FILE = f"GiantModel_{id_to_composer[COMPOSER_ID]}_alpha{ALPHA}_idx{SEED_INDEX}_temp{TEMPERATURE}.mid"
SEED_LENGTH = 1024        # O quanto do começo da música o modelo vai ver

steering_vector = torch.tensor(composer_vectors[COMPOSER_ID], dtype=torch.float32, device=device)
print(f"Vetor de Estilo do compositor {id_to_composer[COMPOSER_ID]} ({COMPOSER_ID}) pronto")

@torch.no_grad()
def generate_music(model, steering_vector, alpha, seed_tokens, max_new_tokens=1024, temperature=1.0, top_k=40, top_p=1.0, repetition_penalty=1.5):
    model.eval()
    generated = seed_tokens.copy()

    for step in range(max_new_tokens):
        context = generated[-(1024 - 1):]
        x = torch.tensor([context], dtype=torch.long, device=device)

        # Steering
        logits = model(x, steering_vector=steering_vector, steering_layer=-1, alpha=alpha)
        logits = logits[0, -1]

        # Penalidade de repetição
        for token in set(generated[-30:]):
            if logits[token] > 0:
                logits[token] /= repetition_penalty
            else:
                logits[token] *= repetition_penalty

        logits = logits / temperature

        # Top-K
        if top_k > 0:
            v, ind = torch.topk(logits, min(top_k, logits.size(-1)))
            logits = torch.full_like(logits, float('-inf'))
            logits[ind] = v

        # Top-P
        probs = torch.softmax(logits, dim=-1)
        sorted_probs, sorted_indices = torch.sort(probs, descending=True)
        cumulative_probs = torch.cumsum(sorted_probs, dim=-1)

        sorted_indices_to_remove = cumulative_probs > top_p
        sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
        sorted_indices_to_remove[..., 0] = 0

        indices_to_remove = sorted_indices[sorted_indices_to_remove]
        probs[indices_to_remove] = 0.0
        probs = probs / probs.sum()

        next_token = torch.multinomial(probs, 1).item()
        generated.append(next_token)

        if (step + 1) % 100 == 0:
            print(f"  -> {step + 1} / {max_new_tokens} tokens gerados")

    return generated

# Execução e salvar

exemplo_midi = midi_paths[SEED_INDEX]
print(f"Extraindo o início real da música base: {exemplo_midi.name}")

tokens_da_musica = tokenizer(exemplo_midi)[0].ids
seed_tokens = tokens_da_musica[:SEED_LENGTH]

generated_tokens = generate_music(
    model=model,
    steering_vector=steering_vector,
    alpha=ALPHA,
    seed_tokens=seed_tokens,
    max_new_tokens=MAX_NEW_TOKENS,
    temperature=TEMPERATURE,
    top_p=1
)

print("\nIniciando conversão estrutural do arquivo (BPE -> REMI)...")
seq = TokSequence(ids=generated_tokens)
seq.are_ids_encoded = True
tokenizer.decode_token_ids(seq)

generation_dir = output_dir / "generated_midis"
generation_dir.mkdir(parents=True, exist_ok=True)

score = tokenizer.decode([seq])
score.dump_midi(generation_dir / OUTPUT_FILE)
print(f"✓ Sucesso! Seu MIDI estilizado foi salvo em: {generation_dir / OUTPUT_FILE}")

SEED_INDEX: 6414 -> Abreu, Zequinha, Tico-Tico no fubá, KyEuFpJuwfc.mid
Vetor de Estilo do compositor Beethoven (112) pronto
Extraindo o início real da música base: Abreu, Zequinha, Tico-Tico no fubá, KyEuFpJuwfc.mid
  -> 100 / 1024 tokens gerados
  -> 200 / 1024 tokens gerados
  -> 300 / 1024 tokens gerados
  -> 400 / 1024 tokens gerados
  -> 500 / 1024 tokens gerados
  -> 600 / 1024 tokens gerados
  -> 700 / 1024 tokens gerados
  -> 800 / 1024 tokens gerados
  -> 900 / 1024 tokens gerados
  -> 1000 / 1024 tokens gerados

Iniciando conversão estrutural do arquivo (BPE -> REMI)...
✓ Sucesso! Seu MIDI estilizado foi salvo em: trabalho_midi_outputs/ComposerVector/generated_midis/GiantModel_Beethoven_alpha0.2_idx6414_temp0.95.mid
